# 01 — Il classificatore a feature (tentativo 13): addestramento, performance e modelli salvati

Il TF-IDF + regressione logistica dei tentativi 06–12, invariato (C tra 0,1, 1, 10 e 100 in validazione interna per
gruppi, pesi per testata, voto per testata), sul dataset solo politici del tentativo 13, etichetta NewsGuard 2026.

In più, per poter spiegare i modelli e usarli in un tool: per ogni fold si salvano **vettorizzatore e modello**
(`../Modelli/feature/fold<k>.joblib`), e alla fine si addestra un **modello finale su tutte le testate**, con il C
scelto più spesso nei fold (`../Modelli/feature/finale.joblib`). Le probabilità per classe di ogni articolo di test
sono in `<NOME>_fold_gruppo_articoli.csv`; per testata e per fold le calcola `03_confronto`.


In [ ]:
import json
from collections import Counter

import joblib
import time

import numpy as np
import scipy.sparse as sp
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, metriche, per_testata, pesi_bilanciati, registratore, LIVELLI, MODELLI

In [ ]:
NOME = "feature_tuned"
ETICHETTA = "2026"          # 2026 (obiettivo) / 2022 (controllo)
TITOLO = False             # True: anche il titolo (TF-IDF a parte, affiancato)
SOLO_POLITICI = False      # True: solo le testate con il solo argomento politico
TAGLIO_TOKEN = 0           # 0 = testo intero; N = primi N token (tokenizer di ModernBERT)
C_GRIGLIA = "0.1,1,10,100" # "" = C fisso a 1; "0.1,1,10,100" = C scelto con una validazione interna per gruppi (solo training)
SCHEMI_DA_FARE = SCHEMI
SOLO_VALUTAZIONE = False   # True: training solo sulle testate valutate (come la prima versione)
N_JOBS = 5                 # fold in parallelo
PROVA = False              # True: poche testate, risultati in Results/3-classificazione/prova/

In [ ]:
NOME = NOME + ("_titolo" if TITOLO else "") + ("_politici" if SOLO_POLITICI else "") + f"_{ETICHETTA}"
OUT = cartella_risultati(PROVA)
DIR_MODELLI = (OUT / "modelli" if PROVA else MODELLI) / "feature"
DIR_MODELLI.mkdir(parents=True, exist_ok=True)
registra = registratore(OUT / f"avanzamento_{NOME}.log")
art = carica(ETICHETTA, PROVA, SOLO_VALUTAZIONE, titolo=TITOLO, solo_politici=SOLO_POLITICI)
if TAGLIO_TOKEN:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    enc = tok(art.text.tolist(), truncation=True, max_length=TAGLIO_TOKEN, add_special_tokens=False, return_offsets_mapping=True)
    art["text"] = [t[:o[-1][1]] if o else t for t, o in zip(art.text, enc["offset_mapping"])]
registra(f"=== {NOME}: {len(art)} articoli, {art.domain.nunique()} testate ({art[art.valutazione].domain.nunique()} valutate), "
         f"taglio {TAGLIO_TOKEN or 'nessuno'}, solo valutate {SOLO_VALUTAZIONE} ===")

## Addestramento

Per ogni fold: `fit_transform` sul training, `transform` sul test. I logit sono la `decision_function` della
regressione logistica.

In [ ]:
def nuovo_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                           max_features=100_000, strip_accents="unicode")


def matrici(a, b):
    """Il TF-IDF dell'articolo (e, con TITOLO, uno a parte sul titolo, affiancato): vocabolari e IDF solo su a."""
    v = nuovo_tfidf(); Xa, Xb = v.fit_transform(a.text), v.transform(b.text)
    w = None
    if TITOLO:
        w = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=30_000, strip_accents="unicode")
        Xa, Xb = sp.hstack([Xa, w.fit_transform(a.titolo)]).tocsr(), sp.hstack([Xb, w.transform(b.titolo)]).tocsr()
    return Xa, Xb, {"testo": v, "titolo": w}


def salva(nome, vett, clf, C, testate):
    """Vettorizzatore e modello, con quello che serve per riusarli (anche in un tool)."""
    joblib.dump({"tfidf": vett["testo"], "tfidf_titolo": vett["titolo"], "modello": clf, "C": C, "classi": LIVELLI,
                 "classi_modello": clf.classes_.tolist(), "testate_training": sorted(testate),
                 "uso": "p = modello.predict_proba(tfidf.transform([testo])); colonne = classi_modello (0 = l1 ... 4 = l5)"},
                DIR_MODELLI / f"{nome}.joblib", compress=3)


def scegli_C(tr):
    """C con la F1 macro per testata (voto) più alta in una validazione a 3 fold per gruppi, sul solo training."""
    griglia = [float(x) for x in str(C_GRIGLIA).split(",")]
    testate = tr.drop_duplicates("domain").reset_index(drop=True)
    punteggi = {c: [] for c in griglia}
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(testate, testate.label, testate.gruppo):
        a = tr[tr.domain.isin(testate.domain[itr])]
        b = tr[tr.domain.isin(testate.domain[iva]) & tr.valutazione]          # si valuta come nel test: solo testate valutate
        Xa, Xb, _ = matrici(a, b)
        pa = pesi_bilanciati(a.label, a.peso)
        for c in griglia:
            p = LogisticRegression(C=c, max_iter=1000, random_state=SEED).fit(Xa, a.label, sample_weight=pa).predict(Xb)
            t = per_testata(pd.DataFrame({"domain": b.domain.values, "y_true": b.label.values, "y_pred": p}))
            punteggi[c].append(f1_score(t.y_true, t.y_pred, average="macro", labels=range(5), zero_division=0))
    medie = {c: float(np.mean(s)) for c, s in punteggi.items()}
    return max(medie, key=medie.get), medie


def un_fold(schema, fold):
    t0 = time.time()
    tr, te = art[art[schema] != fold], art[(art[schema] == fold) & art.valutazione]
    C, medie = (1.0, {}) if not C_GRIGLIA else scegli_C(tr)
    if C_GRIGLIA:
        registra(f"{schema} fold {fold}: C scelto {C} (F1 interne {', '.join(f'{k}: {v:.3f}' for k, v in medie.items())})")
    Xtr, X, vett = matrici(tr, te)                             # vocabolario e IDF: solo training; test: solo transform
    clf = LogisticRegression(C=C, max_iter=1000, random_state=SEED)
    clf.fit(Xtr, tr.label, sample_weight=pesi_bilanciati(tr.label, tr.peso))
    salva(f"fold{fold}", vett, clf, C, tr.domain.unique())
    registra(f"{schema} fold {fold}: fatto in {time.time() - t0:.0f} s ({len(tr)} training, {len(te)} test); modello salvato")
    return te.index.to_numpy(), clf.classes_, clf.decision_function(X), clf.predict_proba(X), C


def addestra(schema):
    file_out = OUT / f"{NOME}_{schema}_articoli.csv"
    if file_out.exists():
        registra(f"{schema}: già fatto, carico {file_out.name}")
        return pd.read_csv(file_out)
    logit, proba = np.full((len(art), 5), np.nan), np.zeros((len(art), 5))
    scelti = []
    for idx, classi, z, p, C in Parallel(n_jobs=N_JOBS)(delayed(un_fold)(schema, f) for f in sorted(art[schema].unique())):
        logit[idx[:, None], classi[None, :]] = z
        proba[idx[:, None], classi[None, :]] = p
        scelti.append(C)
    # il modello finale, su tutte le testate, con il C scelto più spesso nei fold (a parità, il più piccolo)
    conta = Counter(scelti); C = min(c for c in conta if conta[c] == max(conta.values()))
    t0 = time.time(); Xall, _, vett = matrici(art, art.head(1))
    clf = LogisticRegression(C=C, max_iter=1000, random_state=SEED).fit(Xall, art.label, sample_weight=pesi_bilanciati(art.label, art.peso))
    salva("finale", vett, clf, C, art.domain.unique())
    registra(f"modello finale: C {C} (scelti nei fold {scelti}), {len(art)} articoli, {time.time() - t0:.0f} s; salvato")
    pred = pd.DataFrame({"article_id": art.article_id, "domain": art.domain, "fold": art[schema], "y_true": art.label})
    for k in range(5):
        pred[f"logit_l{k + 1}"] = logit[:, k]
    for k, c in enumerate(CLASSI):
        pred[c] = proba[:, k]
    pred["y_pred"] = proba.argmax(1)
    pred = pred[art.valutazione.to_numpy()].reset_index(drop=True)      # solo le testate valutate
    pred.to_csv(file_out, index=False)
    json.dump(metriche(pred), open(OUT / f"{NOME}_{schema}_metriche.json", "w"), indent=1)
    return pred


pred = {s: addestra(s) for s in SCHEMI_DA_FARE}

## Baseline senza testo

Per ogni fold, dalle testate valutate di training (come nella prima versione): la classe più frequente, e una classe estratta a caso con le
proporzioni del training (una per testata, seed fisso).

In [ ]:
def baseline(schema, modo):
    rng = np.random.default_rng(SEED)
    y = np.full(len(art), -1)
    for fold in sorted(art[schema].unique()):
        tr, te = art[(art[schema] != fold) & art.valutazione], art[(art[schema] == fold) & art.valutazione]
        priori = tr.drop_duplicates("domain").label.value_counts(normalize=True).sort_index()
        if modo == "maggioranza":
            y[te.index] = priori.idxmax()
        else:
            scelte = dict(zip(te.domain.unique(), rng.choice(priori.index, te.domain.nunique(), p=priori.values)))
            y[te.index] = te.domain.map(scelte)
    b = pd.DataFrame({"domain": art.domain, "y_true": art.label, "y_pred": y})
    return b[art.valutazione.to_numpy()].reset_index(drop=True)


for s in SCHEMI_DA_FARE:
    for modo in ["maggioranza", "a caso"]:
        json.dump(metriche(baseline(s, modo)), open(OUT / f"baseline_{modo.replace(' ', '_')}{'_politici' if SOLO_POLITICI else ''}_{ETICHETTA}_{s}_metriche.json", "w"), indent=1)

## Risultati

Per articolo e per testata (voto di maggioranza): accuratezza a 5 classi, F1 macro, binario e tolerant.

In [ ]:
righe = []
for s in SCHEMI_DA_FARE:
    righe.append({"modello": f"TF-IDF + LR ({NOME})", "fold": NOME_SCHEMA[s], **letture(pred[s])})
    for modo in ["maggioranza", "a caso"]:
        righe.append({"modello": f"baseline: {modo}", "fold": NOME_SCHEMA[s], **letture(baseline(s, modo))})
ris = pd.DataFrame(righe).set_index(["fold", "modello"])
ris.to_csv(OUT / f"{NOME}_metriche.csv")
ris[[c for c in ris.columns if c.startswith("testata")] + ["articolo 5cl", "articolo bin"]]

In [ ]:
t = per_testata(pred[SCHEMI_DA_FARE[0]])
LIV = LIVELLI
display(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]))
(OUT / f"{NOME}_FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))